###imports

In [1]:
import pandas as pd
import json
from pathlib import Path
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

###data


In [3]:
df_department = pd.read_csv("/content/department-v2.csv")
df_seniority = pd.read_csv("/content/seniority-v2.csv")

In [4]:
def workers_positions_to_csv(data):
    people = json.loads(Path(data).read_text(encoding="utf-8"))
    rows = []

    for person_idx, jobs in enumerate(people, start=1):
        jobs_sorted = sorted(jobs, key=lambda d: (d.get("startDate") is None, d.get("startDate")))
        for job_idx, j in enumerate(jobs_sorted, start=1):
            rows.append({
                "person_id": person_idx,
                "job_no": job_idx,
                "organization": j.get("organization"),
                "linkedin": j.get("linkedin"),
                "position": j.get("position"),
                "startDate": j.get("startDate"),
                "endDate": j.get("endDate"),
                "status": j.get("status"),
                "department": j.get("department"),
                "seniority": j.get("seniority"),
            })

    df = pd.DataFrame(rows, columns=[
        "person_id","job_no","organization","linkedin","position",
        "startDate","endDate","status","department","seniority"
    ])
    df["person_id"] = df["person_id"].astype(int)
    df["job_no"] = df["job_no"].astype(int)
    return df

In [5]:
df_t_annotated= workers_positions_to_csv("/content/linkedin-cvs-annotated.json")
df_t_notannotated= workers_positions_to_csv("/content/linkedin-cvs-not-annotated.json")

#Approach

**Embedding-based classification (two-step pipeline)**

* I converted each job title (text / position) into a dense vector using a SentenceTransformer (with normalized embeddings).

* I then trained a supervised classifier on top of these embeddings using the labels from department-v2.csv / seniority-v2.csv, using a stratified train/test split.

**Classifier choice: Logistic Regression vs. Random Forest**

* I tested Logistic Regression and Random Forest as the classifier on top of the embeddings.

* Logistic Regression worked better, so I dropped the Random Forest Classifier for further runs.

**Embedding model selection**

* I compared multiple embedding models (choosen the top models from this list: https://huggingface.co/sentence-transformers/models?sort=downloads).

* The best overall results (for both department and seniority experiments) came from paraphrase-multilingual-mpnet-base-v2, so I chose this model for further configuration.

**Train/test split experiments**

* I tried different stratified splits (e.g. 20/80, 40/60, 50/50, 60/40).

* A larger test split gave more stable evaluation maybe because otherwise some rare labels end up with very few test examples (e.g., “Other” can drop below 10)

**Inference-processing for catch-all/unseen labels**

* For low-confidence predictions, I experimented with a probability threshold to
assign the catch-all/unseen labels more often (e.g., “Other” for department / “Professional” for seniority) by overwritting predictions on ambiguous titles.

**Normalization**

* I used normalization in the ebeddings to scale each embedding vector to length 1. Without this normalization, embedding magnitudes can skew similarity and classifier behavior, so longer titles may appear more or less similar than they actually are.

#Results


##Department
paraphrase-multilingual-MiniLM-L12-v2
- department-v2 train/test-split 80/20, Logistic Regression
  - Acc.: 0.3590, Recall avg: 0.5135
- train/test-split 80/20,Random Forest
  Acc.: 0.2752, Recall avg: 0.3733

all-MiniLM-L6-v2  
- train/test-split 80/20, Logistic Regression
  - Acc.: 0.3692, Recall avg: 0.4850
- train/test-split 80/20,Random Forest
  - Acc.: 0.2233, Recall avg: 0.3181

all-mpnet-base-v2  
- , train/test-split 80/20, Logistic Regression
  - Acc.: 0.3791, Recall avg: 0.5134

paraphrase-multilingual-mpnet-base-v2
- train/test-split 80/20, Logistic Regression
  - without threshold: Acc.: 0.3965, Recall avg: 0.5515
  - with threshold (0.50): Acc.: 0.5796, Recall avg: 0.4888
- train/test-split 60/40, Logistic Regression
  - without threshold: Acc.: 0.4041, Recall avg: 0.5497
  - with threshold (0.50): Acc.: 0.5902, Recall avg: 0.4774
- **train/test-split 50/50, Logistic Regression**
  - without threshold: Acc.: 0.4105, Recall avg: 0.5555
  - **with threshold (0.50): Acc.: 0.5963 , Recall avg: 0.4789**
- train/test-split 40/60, Logistic Regression
  - without threshold: Acc.: 0.4079, Recall avg: 0.5459
  - with threshold (0.50): Acc.: 0.5967, Recall avg: 0.4622



---


##Seniority
paraphrase-multilingual-MiniLM-L12-v2
- train/test-split 80/20, Logistic Regression
  - Acc.: 0.3745, Recall avg: 0.6051

paraphrase-multilingual-mpnet-base-v2
- train/test-split 80/20, Logistic Regression
  - without threshold: Acc.: 0.3992, Recall avg: 0.6496
  - with threshold (0.60): Acc.: 0.5193, Recall avg: 0.6425
- **train/test-split 50/50, Logistic Regression**
  - without threshold: Acc.: 0.3999, Recall avg: 0.6505
  - **with threshold (0.60): Acc.: 0.5402, Recall avg: 0.6391**
- train/test-split 40/60, Logistic Regression
  - without threshold: Acc.: 0.4011, Recall avg: 0.6522
  - with threshold (0.60): Acc.: 0.5368, Recall avg: 0.6292




#Models

#department model

##paraphrase-multilingual-MiniLM-L12-v2

In [6]:
df = df_department
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_1_LR = LogisticRegression(class_weight="balanced")
clf_1_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_1_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (4 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

v2 eval accuracy: 0.8965
                        precision    recall  f1-score   support

        Administrative     0.2766    0.7647    0.4062        17
  Business Development     0.8511    0.9677    0.9057       124
            Consulting     0.5882    0.9091    0.7143        33
      Customer Support     0.2143    0.8571    0.3429         7
       Human Resources     0.8571    1.0000    0.9231         6
Information Technology     0.9091    0.8429    0.8748       261
             Marketing     0.9788    0.9115    0.9439       859
                 Other     0.6667    1.0000    0.8000         8
    Project Management     0.5312    0.8500    0.6538        40
            Purchasing     0.7778    0.8750    0.8235         8
                 Sales     0.9427    0.8889    0.9150       666

              accuracy                         0.8965      2029
             macro avg     0.6903    0.8970    0.7548      2029
          weighted avg     0.9241    0.8965    0.9061      2029



In [7]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)

# --- predict ---
y_pred = clf_1_LR.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_1_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.359
                        precision    recall  f1-score   support

        Administrative     0.1205    0.4762    0.1923        84
  Business Development     0.2752    0.5256    0.3612        78
            Consulting     0.3041    0.7282    0.4290       195
      Customer Support     0.2727    0.4375    0.3360        48
       Human Resources     0.5185    0.6000    0.5563        70
Information Technology     0.3280    0.3974    0.3594       312
             Marketing     0.4929    0.5188    0.5055       133
                 Other     0.6831    0.1326    0.2221      1252
    Project Management     0.4161    0.6800    0.5163       175
            Purchasing     0.3656    0.4722    0.4121        72
                 Sales     0.3801    0.6804    0.4877       219

              accuracy                         0.3590      2638
             macro avg     0.3779    0.5135    0.3980      2638
          weighted avg     0.5102    0.3590    0.3288      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,40,2,1,1,2,8,1,8,3,9,9
Business Development,1,41,21,0,0,1,1,7,2,0,4
Consulting,8,13,142,5,5,5,4,1,9,0,3
Customer Support,3,2,2,21,1,2,1,7,0,0,9
Human Resources,8,0,10,0,42,1,3,3,1,0,2
Information Technology,5,9,44,29,2,124,1,33,58,0,7
Marketing,9,6,10,1,2,13,69,2,14,2,5
Other,238,63,226,14,26,199,42,166,69,41,168
Project Management,0,5,6,1,0,11,4,5,119,0,24
Purchasing,5,0,3,2,0,2,2,5,7,34,12


###Random Forest

In [8]:
# training RF classifier on embeddings

clf_1_RF = RandomForestClassifier(
    n_estimators=1000,
    class_weight="balanced_subsample",
    random_state=42,
)
clf_1_RF.fit(X_train_emb, y_train)
y_pred = clf_1_RF.predict(X_eval_emb)


In [9]:
# --- predict ---
y_pred = clf_1_RF.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_1_RF.classes_) | set(y_true))

print("Random Forest accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Random Forest accuracy: 0.2741
                        precision    recall  f1-score   support

        Administrative     0.7778    0.1667    0.2745        84
  Business Development     0.3627    0.4744    0.4111        78
            Consulting     0.8384    0.4256    0.5646       195
      Customer Support     0.0000    0.0000    0.0000        48
       Human Resources     1.0000    0.2571    0.4091        70
Information Technology     0.5675    0.5929    0.5799       312
             Marketing     0.2029    0.7293    0.3175       133
                 Other     0.7778    0.0056    0.0111      1252
    Project Management     0.9195    0.4571    0.6107       175
            Purchasing     0.8889    0.1111    0.1975        72
                 Sales     0.1301    0.8858    0.2269       219

              accuracy                         0.2741      2638
             macro avg     0.5878    0.3733    0.3275      2638
          weighted avg     0.6665    0.2741    0.2281      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,14,2,0,0,0,8,14,1,0,0,45
Business Development,0,37,0,0,0,0,12,0,0,0,29
Consulting,0,14,83,0,0,18,44,0,3,0,33
Customer Support,0,0,1,0,0,6,3,0,0,0,38
Human Resources,0,0,1,0,18,3,14,0,0,0,34
Information Technology,0,5,1,1,0,185,15,1,0,0,104
Marketing,3,2,0,0,0,8,97,0,1,0,22
Other,1,36,13,0,0,75,244,7,2,1,873
Project Management,0,4,0,0,0,18,13,0,80,0,60
Purchasing,0,0,0,0,0,0,5,0,0,8,59


##all-MiniLM-L6-v2

In [10]:
df = df_department
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_2_LR = LogisticRegression(class_weight="balanced")
clf_2_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_2_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

v2 eval accuracy: 0.9236
                        precision    recall  f1-score   support

        Administrative     0.6071    1.0000    0.7556        17
  Business Development     0.8288    0.9758    0.8963       124
            Consulting     0.6889    0.9394    0.7949        33
      Customer Support     0.7778    1.0000    0.8750         7
       Human Resources     0.7143    0.8333    0.7692         6
Information Technology     0.8947    0.9119    0.9032       261
             Marketing     0.9854    0.9430    0.9637       859
                 Other     0.4706    1.0000    0.6400         8
    Project Management     0.5286    0.9250    0.6727        40
            Purchasing     0.6667    0.7500    0.7059         8
                 Sales     0.9738    0.8919    0.9310       666

              accuracy                         0.9236      2029
             macro avg     0.7397    0.9246    0.8098      2029
          weighted avg     0.9385    0.9236    0.9277      2029



In [11]:
df = df_department
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_2_LR = LogisticRegression(class_weight="balanced")
clf_2_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_2_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))


Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

v2 eval accuracy: 0.9236
                        precision    recall  f1-score   support

        Administrative     0.6071    1.0000    0.7556        17
  Business Development     0.8288    0.9758    0.8963       124
            Consulting     0.6889    0.9394    0.7949        33
      Customer Support     0.7778    1.0000    0.8750         7
       Human Resources     0.7143    0.8333    0.7692         6
Information Technology     0.8947    0.9119    0.9032       261
             Marketing     0.9854    0.9430    0.9637       859
                 Other     0.4706    1.0000    0.6400         8
    Project Management     0.5286    0.9250    0.6727        40
            Purchasing     0.6667    0.7500    0.7059         8
                 Sales     0.9738    0.8919    0.9310       666

              accuracy                         0.9236      2029
             macro avg     0.7397    0.9246    0.8098      2029
          weighted avg     0.9385    0.9236    0.9277      2029



In [12]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(
    final_eval["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
    )

# --- predict ---
y_pred = clf_2_LR.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_2_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.3692
                        precision    recall  f1-score   support

        Administrative     0.0901    0.1190    0.1026        84
  Business Development     0.2897    0.5385    0.3767        78
            Consulting     0.3376    0.6769    0.4505       195
      Customer Support     0.5278    0.3958    0.4524        48
       Human Resources     0.5846    0.5429    0.5630        70
Information Technology     0.2519    0.5385    0.3432       312
             Marketing     0.4676    0.4887    0.4779       133
                 Other     0.7206    0.1422    0.2375      1252
    Project Management     0.4599    0.7543    0.5714       175
            Purchasing     0.4328    0.4028    0.4173        72
                 Sales     0.3333    0.7352    0.4587       219

              accuracy                         0.3692      2638
             macro avg     0.4087    0.4850    0.4046      2638
          weighted avg     0.5269    0.3692    0.3357      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,10,5,3,0,1,17,2,10,3,3,30
Business Development,0,42,16,0,0,7,3,4,0,0,6
Consulting,3,21,132,1,2,13,5,0,8,1,9
Customer Support,0,2,4,19,0,5,2,7,0,0,9
Human Resources,3,1,7,1,38,10,1,4,1,1,3
Information Technology,0,10,75,4,1,168,1,18,17,1,17
Marketing,2,1,8,0,1,32,65,1,11,2,10
Other,87,57,132,10,22,371,45,178,102,27,221
Project Management,1,5,6,0,0,16,1,6,132,0,8
Purchasing,0,0,5,0,0,14,0,4,11,29,9


###Random forest

In [13]:
# training RF classifier on embeddings

clf_2_RF = RandomForestClassifier(
    n_estimators=1000,
    class_weight="balanced_subsample",
    random_state=42,
    )
clf_2_RF.fit(X_train_emb, y_train)
y_pred = clf_2_RF.predict(X_eval_emb)


In [14]:
# --- predict ---
y_pred = clf_2_RF.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_2_RF.classes_) | set(y_true))

print("Random Forest accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Random Forest accuracy: 0.2233
                        precision    recall  f1-score   support

        Administrative     0.8750    0.0833    0.1522        84
  Business Development     0.4833    0.3718    0.4203        78
            Consulting     0.9846    0.3282    0.4923       195
      Customer Support     0.7500    0.0625    0.1154        48
       Human Resources     1.0000    0.3286    0.4946        70
Information Technology     0.6752    0.3397    0.4520       312
             Marketing     0.2787    0.6391    0.3881       133
                 Other     1.0000    0.0024    0.0048      1252
    Project Management     0.9836    0.3429    0.5085       175
            Purchasing     1.0000    0.0694    0.1299        72
                 Sales     0.1048    0.9315    0.1884       219

              accuracy                         0.2233      2638
             macro avg     0.7396    0.3181    0.3042      2638
          weighted avg     0.8249    0.2233    0.1971      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,7,2,0,0,0,1,0,0,0,0,74
Business Development,0,29,0,0,0,0,7,0,0,0,42
Consulting,0,10,64,0,0,9,20,0,1,0,91
Customer Support,0,0,0,3,0,3,6,0,0,0,36
Human Resources,0,0,1,0,23,1,4,0,0,0,41
Information Technology,0,6,0,1,0,106,16,0,0,0,183
Marketing,0,0,0,0,0,8,85,0,0,0,40
Other,1,10,0,0,0,16,127,3,0,0,1095
Project Management,0,3,0,0,0,11,19,0,60,0,82
Purchasing,0,0,0,0,0,1,7,0,0,5,59


##all-mpnet-base-v2

In [15]:
df = df_department
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("all-mpnet-base-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_3_LR = LogisticRegression(class_weight="balanced")
clf_3_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_3_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

... (4 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

v2 eval accuracy: 0.9177
                        precision    recall  f1-score   support

        Administrative     0.5926    0.9412    0.7273        17
  Business Development     0.8741    0.9516    0.9112       124
            Consulting     0.7273    0.9697    0.8312        33
      Customer Support     0.5000    0.5714    0.5333         7
       Human Resources     0.8571    1.0000    0.9231         6
Information Technology     0.8764    0.9234    0.8993       261
             Marketing     0.9863    0.9232    0.9537       859
                 Other     0.4444    1.0000    0.6154         8
    Project Management     0.5068    0.9250    0.6549        40
            Purchasing     0.7273    1.0000    0.8421         8
                 Sales     0.9553    0.8994    0.9265       666

              accuracy                         0.9177      2029
             macro avg     0.7316    0.9186    0.8016      2029
          weighted avg     0.9330    0.9177    0.9221      2029



In [16]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)


# --- predict ---
y_pred = clf_3_LR.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_3_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.3791
                        precision    recall  f1-score   support

        Administrative     0.1493    0.1190    0.1325        84
  Business Development     0.3188    0.5641    0.4074        78
            Consulting     0.3158    0.7385    0.4424       195
      Customer Support     0.5400    0.5625    0.5510        48
       Human Resources     0.6500    0.5571    0.6000        70
Information Technology     0.3409    0.4359    0.3826       312
             Marketing     0.3860    0.4962    0.4342       133
                 Other     0.7238    0.1653    0.2692      1252
    Project Management     0.3597    0.7543    0.4871       175
            Purchasing     0.2826    0.5417    0.3714        72
                 Sales     0.3083    0.7123    0.4303       219

              accuracy                         0.3791      2638
             macro avg     0.3977    0.5134    0.4098      2638
          weighted avg     0.5250    0.3791    0.3480      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,10,5,3,0,0,16,3,15,7,6,19
Business Development,1,44,14,1,1,3,1,6,2,0,5
Consulting,4,22,144,1,1,4,4,2,10,0,3
Customer Support,1,2,7,27,0,1,0,5,0,1,4
Human Resources,5,1,10,0,39,2,1,4,3,1,4
Information Technology,1,11,56,15,2,136,3,23,35,2,28
Marketing,2,4,8,0,0,26,66,4,16,3,4
Other,38,42,194,5,15,188,70,207,145,77,271
Project Management,0,5,7,0,0,11,8,3,132,2,7
Purchasing,1,0,8,0,0,2,1,6,10,39,5


##paraphrase-multilingual-mpnet-base-v2

###Tariningsplit 20/80

In [17]:
df = df_department
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_4_LR = LogisticRegression(class_weight="balanced")
clf_4_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_4_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

... (4 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

v2 eval accuracy: 0.9147
                        precision    recall  f1-score   support

        Administrative     0.4054    0.8824    0.5556        17
  Business Development     0.8855    0.9355    0.9098       124
            Consulting     0.6667    0.9697    0.7901        33
      Customer Support     0.5714    0.5714    0.5714         7
       Human Resources     0.8571    1.0000    0.9231         6
Information Technology     0.8881    0.9119    0.8998       261
             Marketing     0.9825    0.9173    0.9488       859
                 Other     0.6667    1.0000    0.8000         8
    Project Management     0.5385    0.8750    0.6667        40
            Purchasing     0.5455    0.7500    0.6316         8
                 Sales     0.9485    0.9129    0.9304       666

              accuracy                         0.9147      2029
             macro avg     0.7233    0.8842    0.7843      2029
          weighted avg     0.9298    0.9147    0.9194      2029



In [18]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)


# --- predict ---
y_pred = clf_4_LR.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_4_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.3965
                        precision    recall  f1-score   support

        Administrative     0.1135    0.4405    0.1805        84
  Business Development     0.2204    0.5256    0.3106        78
            Consulting     0.3089    0.7795    0.4425       195
      Customer Support     0.4667    0.4375    0.4516        48
       Human Resources     0.4112    0.6286    0.4972        70
Information Technology     0.4785    0.5353    0.5053       312
             Marketing     0.6091    0.5038    0.5514       133
                 Other     0.7699    0.1470    0.2468      1252
    Project Management     0.3290    0.7314    0.4539       175
            Purchasing     0.4624    0.5972    0.5212        72
                 Sales     0.5364    0.7397    0.6219       219

              accuracy                         0.3965      2638
             macro avg     0.4278    0.5515    0.4348      2638
          weighted avg     0.5840    0.3965    0.3697      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,37,3,2,1,1,11,0,12,3,6,8
Business Development,0,41,19,0,1,0,3,7,2,0,5
Consulting,5,13,152,1,4,5,5,1,6,0,3
Customer Support,5,2,7,21,0,3,0,3,0,0,7
Human Resources,9,0,10,0,44,1,1,2,2,1,0
Information Technology,2,14,38,15,3,167,0,13,57,0,3
Marketing,7,1,10,2,2,17,67,3,18,2,4
Other,250,100,240,5,51,120,23,184,160,33,86
Project Management,0,6,4,0,1,13,3,4,128,0,16
Purchasing,5,0,1,0,0,5,0,4,6,43,8


###threshold to predict "Other" better

In [19]:
OTHER_LABEL = "Other"

proba = clf_4_LR.predict_proba(X_final_eval_emb)
classes = clf_4_LR.classes_

best_idx = proba.argmax(axis=1)
best_p = proba.max(axis=1)
pred = classes[best_idx]

THRESHOLD= 0.50
y_pred_thr = np.where(best_p < THRESHOLD, OTHER_LABEL, pred)

print("accuracy:", round(accuracy_score(y_true, y_pred_thr), 4))
print(classification_report(y_true, y_pred_thr, digits=4, zero_division=0))

accuracy: 0.5796
                        precision    recall  f1-score   support

        Administrative     0.1983    0.2857    0.2341        84
  Business Development     0.3846    0.5128    0.4396        78
            Consulting     0.4308    0.7026    0.5341       195
      Customer Support     0.7619    0.3333    0.4638        48
       Human Resources     0.6316    0.5143    0.5669        70
Information Technology     0.6855    0.2724    0.3899       312
             Marketing     0.7590    0.4737    0.5833       133
                 Other     0.6148    0.6973    0.6534      1252
    Project Management     0.5846    0.6514    0.6162       175
            Purchasing     0.6200    0.4306    0.5082        72
                 Sales     0.7586    0.5023    0.6044       219

              accuracy                         0.5796      2638
             macro avg     0.5845    0.4888    0.5085      2638
          weighted avg     0.6100    0.5796    0.5740      2638



###Trainingsplit 40/60

In [20]:
df = df_department
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.40,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_4_2_LR = LogisticRegression(class_weight="balanced")
clf_4_2_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_4_2_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

Batches:   0%|          | 0/191 [00:00<?, ?it/s]

Batches:   0%|          | 0/127 [00:00<?, ?it/s]

v2 eval accuracy: 0.915
                        precision    recall  f1-score   support

        Administrative     0.4677    0.8788    0.6105        33
  Business Development     0.8726    0.9113    0.8915       248
            Consulting     0.6742    0.8955    0.7692        67
      Customer Support     0.4400    0.8462    0.5789        13
       Human Resources     0.7222    1.0000    0.8387        13
Information Technology     0.8976    0.8736    0.8854       522
             Marketing     0.9828    0.9313    0.9564      1718
                 Other     0.6071    1.0000    0.7556        17
    Project Management     0.5105    0.9125    0.6547        80
            Purchasing     0.5882    0.6250    0.6061        16
                 Sales     0.9508    0.9151    0.9326      1331

              accuracy                         0.9150      4058
             macro avg     0.7013    0.8899    0.7709      4058
          weighted avg     0.9303    0.9150    0.9198      4058



In [21]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)


# --- predict ---
y_pred = clf_4_2_LR.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_4_2_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.4041
                        precision    recall  f1-score   support

        Administrative     0.1258    0.4643    0.1980        84
  Business Development     0.2531    0.5256    0.3417        78
            Consulting     0.3141    0.7538    0.4434       195
      Customer Support     0.4000    0.4583    0.4272        48
       Human Resources     0.3898    0.6571    0.4894        70
Information Technology     0.4516    0.4936    0.4717       312
             Marketing     0.6216    0.5188    0.5656       133
                 Other     0.8087    0.1789    0.2930      1252
    Project Management     0.3157    0.7143    0.4378       175
            Purchasing     0.4878    0.5556    0.5195        72
                 Sales     0.5000    0.7260    0.5922       219

              accuracy                         0.4041      2638
             macro avg     0.4244    0.5497    0.4345      2638
          weighted avg     0.5966    0.4041    0.3857      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,39,2,2,1,1,9,0,11,7,4,8
Business Development,0,41,18,0,1,0,3,7,3,0,5
Consulting,4,13,147,1,7,7,5,1,7,0,3
Customer Support,5,2,6,22,0,2,1,3,0,0,7
Human Resources,9,0,9,0,46,1,0,2,2,1,0
Information Technology,3,14,42,21,3,154,0,14,57,1,3
Marketing,7,2,9,2,2,17,69,2,18,2,3
Other,233,79,222,7,57,125,19,224,159,25,102
Project Management,0,4,3,1,1,14,4,4,125,0,19
Purchasing,4,0,1,0,0,5,0,4,9,40,9


###threshold to predict "Other" better

In [22]:
OTHER_LABEL = "Other"

proba = clf_4_2_LR.predict_proba(X_final_eval_emb)
classes = clf_4_2_LR.classes_

best_idx = proba.argmax(axis=1)
best_p = proba.max(axis=1)
pred = classes[best_idx]

THRESHOLD= 0.50
y_pred_thr = np.where(best_p < THRESHOLD, OTHER_LABEL, pred)

print("accuracy:", round(accuracy_score(y_true, y_pred_thr), 4))
print(classification_report(y_true, y_pred_thr, digits=4, zero_division=0))

accuracy: 0.5902
                        precision    recall  f1-score   support

        Administrative     0.2330    0.2857    0.2567        84
  Business Development     0.3922    0.5128    0.4444        78
            Consulting     0.4669    0.6513    0.5439       195
      Customer Support     0.7273    0.3333    0.4571        48
       Human Resources     0.6316    0.5143    0.5669        70
Information Technology     0.6944    0.2404    0.3571       312
             Marketing     0.7750    0.4662    0.5822       133
                 Other     0.6188    0.7468    0.6768      1252
    Project Management     0.5477    0.6229    0.5829       175
            Purchasing     0.6744    0.4028    0.5043        72
                 Sales     0.7376    0.4749    0.5778       219

              accuracy                         0.5902      2638
             macro avg     0.5908    0.4774    0.5046      2638
          weighted avg     0.6144    0.5902    0.5781      2638



###Trainingsplit 50/50

In [23]:
df = df_department
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.50,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_4_3_LR = LogisticRegression(class_weight="balanced")
clf_4_3_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_4_3_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

Batches:   0%|          | 0/159 [00:00<?, ?it/s]

Batches:   0%|          | 0/159 [00:00<?, ?it/s]

v2 eval accuracy: 0.9113
                        precision    recall  f1-score   support

        Administrative     0.4458    0.9024    0.5968        41
  Business Development     0.8871    0.9129    0.8998       310
            Consulting     0.6726    0.9048    0.7716        84
      Customer Support     0.4138    0.7500    0.5333        16
       Human Resources     0.6667    0.9333    0.7778        15
Information Technology     0.8884    0.8775    0.8829       653
             Marketing     0.9802    0.9227    0.9506      2148
                 Other     0.6250    0.9524    0.7547        21
    Project Management     0.5291    0.9010    0.6667       101
            Purchasing     0.6364    0.7000    0.6667        20
                 Sales     0.9418    0.9141    0.9277      1664

              accuracy                         0.9113      5073
             macro avg     0.6988    0.8792    0.7662      5073
          weighted avg     0.9262    0.9113    0.9160      5073



In [24]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)


# --- predict ---
y_pred = clf_4_3_LR.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_4_3_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.4105
                        precision    recall  f1-score   support

        Administrative     0.1216    0.4762    0.1937        84
  Business Development     0.2515    0.5256    0.3402        78
            Consulting     0.3136    0.7590    0.4438       195
      Customer Support     0.4444    0.4167    0.4301        48
       Human Resources     0.4327    0.6429    0.5172        70
Information Technology     0.4592    0.5417    0.4971       312
             Marketing     0.6372    0.5414    0.5854       133
                 Other     0.8125    0.1765    0.2900      1252
    Project Management     0.3432    0.7257    0.4661       175
            Purchasing     0.4773    0.5833    0.5250        72
                 Sales     0.5032    0.7215    0.5929       219

              accuracy                         0.4105      2638
             macro avg     0.4360    0.5555    0.4438      2638
          weighted avg     0.6037    0.4105    0.3910      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,40,3,3,0,2,10,0,11,6,4,5
Business Development,0,41,19,0,0,0,3,6,2,0,7
Consulting,5,13,148,1,6,5,5,1,8,0,3
Customer Support,6,3,5,20,0,3,1,3,0,0,7
Human Resources,8,0,11,0,45,1,0,2,2,1,0
Information Technology,3,14,31,19,4,169,0,12,55,1,4
Marketing,5,2,9,0,2,18,72,2,18,2,3
Other,249,78,234,5,44,137,18,221,140,29,97
Project Management,0,4,2,0,1,15,4,2,127,0,20
Purchasing,7,0,1,0,0,2,0,4,6,42,10


###threshold to predict "Other" better

In [25]:
OTHER_LABEL = "Other"

proba = clf_4_3_LR.predict_proba(X_final_eval_emb)
classes = clf_4_3_LR.classes_

best_idx = proba.argmax(axis=1)
best_p = proba.max(axis=1)
pred = classes[best_idx]

THRESHOLD= 0.50
y_pred_thr = np.where(best_p < THRESHOLD, OTHER_LABEL, pred)

print("accuracy:", round(accuracy_score(y_true, y_pred_thr), 4))
print(classification_report(y_true, y_pred_thr, digits=4, zero_division=0))

accuracy: 0.5963
                        precision    recall  f1-score   support

        Administrative     0.2500    0.2976    0.2717        84
  Business Development     0.3883    0.5128    0.4420        78
            Consulting     0.4691    0.6615    0.5489       195
      Customer Support     0.6957    0.3333    0.4507        48
       Human Resources     0.7400    0.5286    0.6167        70
Information Technology     0.6727    0.2372    0.3507       312
             Marketing     0.7662    0.4436    0.5619       133
                 Other     0.6122    0.7628    0.6792      1252
    Project Management     0.6585    0.6171    0.6372       175
            Purchasing     0.6818    0.4167    0.5172        72
                 Sales     0.7576    0.4566    0.5698       219

              accuracy                         0.5963      2638
             macro avg     0.6084    0.4789    0.5133      2638
          weighted avg     0.6203    0.5963    0.5827      2638



###Trainingsplit 60/40

In [26]:
df = df_department
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.60,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_4_4_LR = LogisticRegression(class_weight="balanced")
clf_4_4_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_4_4_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

Batches:   0%|          | 0/127 [00:00<?, ?it/s]

Batches:   0%|          | 0/191 [00:00<?, ?it/s]

v2 eval accuracy: 0.9069
                        precision    recall  f1-score   support

        Administrative     0.4792    0.9200    0.6301        50
  Business Development     0.8543    0.9140    0.8831       372
            Consulting     0.6383    0.9000    0.7469       100
      Customer Support     0.3947    0.7500    0.5172        20
       Human Resources     0.7500    0.9474    0.8372        19
Information Technology     0.8779    0.8633    0.8706       783
             Marketing     0.9815    0.9263    0.9531      2577
                 Other     0.6571    0.9200    0.7667        25
    Project Management     0.4909    0.9000    0.6353       120
            Purchasing     0.6429    0.7500    0.6923        24
                 Sales     0.9444    0.9009    0.9221      1997

              accuracy                         0.9069      6087
             macro avg     0.7010    0.8811    0.7686      6087
          weighted avg     0.9235    0.9069    0.9121      6087



In [27]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)


# --- predict ---
y_pred = clf_4_4_LR.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_4_4_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.4079
                        precision    recall  f1-score   support

        Administrative     0.1059    0.4286    0.1698        84
  Business Development     0.2515    0.5385    0.3429        78
            Consulting     0.3304    0.7641    0.4613       195
      Customer Support     0.3929    0.4583    0.4231        48
       Human Resources     0.3543    0.6429    0.4569        70
Information Technology     0.5032    0.5096    0.5064       312
             Marketing     0.6283    0.5338    0.5772       133
                 Other     0.8436    0.1853    0.3039      1252
    Project Management     0.3342    0.7314    0.4588       175
            Purchasing     0.4000    0.5000    0.4444        72
                 Sales     0.4875    0.7123    0.5788       219

              accuracy                         0.4079      2638
             macro avg     0.4211    0.5459    0.4294      2638
          weighted avg     0.6169    0.4079    0.3933      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,36,3,3,1,3,11,0,10,6,5,6
Business Development,0,42,19,0,0,0,4,5,2,0,6
Consulting,7,14,149,0,6,5,3,1,7,0,3
Customer Support,5,3,3,22,0,3,0,3,0,0,9
Human Resources,9,0,9,0,45,1,1,2,1,2,0
Information Technology,2,11,32,25,4,159,0,11,64,0,4
Marketing,4,3,10,0,3,18,71,1,18,2,3
Other,265,79,212,8,65,92,21,232,145,37,96
Project Management,0,5,3,0,1,12,3,2,128,0,21
Purchasing,5,0,2,0,0,6,0,3,4,36,16


###threshold to predict "Other" better

In [28]:
OTHER_LABEL = "Other"

proba = clf_4_4_LR.predict_proba(X_final_eval_emb)
classes = clf_4_4_LR.classes_

best_idx = proba.argmax(axis=1)
best_p = proba.max(axis=1)
pred = classes[best_idx]

THRESHOLD= 0.50
y_pred_thr = np.where(best_p < THRESHOLD, OTHER_LABEL, pred)

print("accuracy:", round(accuracy_score(y_true, y_pred_thr), 4))
print(classification_report(y_true, y_pred_thr, digits=4, zero_division=0))

accuracy: 0.5967
                        precision    recall  f1-score   support

        Administrative     0.2283    0.2500    0.2386        84
  Business Development     0.3939    0.5000    0.4407        78
            Consulting     0.5234    0.6872    0.5942       195
      Customer Support     0.6364    0.2917    0.4000        48
       Human Resources     0.6607    0.5286    0.5873        70
Information Technology     0.7143    0.2083    0.3226       312
             Marketing     0.7703    0.4286    0.5507       133
                 Other     0.6048    0.7812    0.6818      1252
    Project Management     0.6562    0.6000    0.6269       175
            Purchasing     0.6190    0.3611    0.4561        72
                 Sales     0.7597    0.4475    0.5632       219

              accuracy                         0.5967      2638
             macro avg     0.5970    0.4622    0.4966      2638
          weighted avg     0.6206    0.5967    0.5777      2638



#seniority model

##paraphrase-multilingual-MiniLM-L12-v2

In [29]:
df = df_seniority
X_all = df_seniority["text"]
y_all = df_seniority["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_sen_LR = LogisticRegression(class_weight="balanced")
clf_sen_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_sen_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

Batches:   0%|          | 0/236 [00:00<?, ?it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

v2 eval accuracy: 0.8807
              precision    recall  f1-score   support

    Director     0.9350    0.9492    0.9421       197
      Junior     0.5891    0.9268    0.7204        82
        Lead     0.9071    0.8815    0.8941       709
  Management     0.6885    0.8344    0.7545       151
      Senior     0.9445    0.8661    0.9036       747

    accuracy                         0.8807      1886
   macro avg     0.8129    0.8916    0.8429      1886
weighted avg     0.8935    0.8807    0.8842      1886



In [30]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "seniority"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)


# --- predict ---
y_pred = clf_sen_LR.predict(X_final_eval_emb)
y_true = final_eval["seniority"].tolist()

# --- results ---
all_labels = sorted(set(clf_sen_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.3745
              precision    recall  f1-score   support

    Director     0.6933    0.8014    0.7434       141
      Junior     0.2852    0.7304    0.4103       230
        Lead     0.5612    0.5978    0.5789       460
  Management     0.6107    0.7129    0.6578       418
Professional     0.0000    0.0000    0.0000      1219
      Senior     0.1476    0.7882    0.2486       170

    accuracy                         0.3745      2638
   macro avg     0.3830    0.6051    0.4398      2638
weighted avg     0.2661    0.3745    0.2967      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,113,2,7,13,0,6
Junior,1,168,15,4,0,42
Lead,3,19,275,58,0,105
Management,35,23,29,298,0,33
Professional,9,352,159,111,0,588
Senior,2,25,5,4,0,134


###threshold to predict "Professional"

In [31]:
OTHER_LABEL = "Professional"

proba = clf_sen_LR.predict_proba(X_final_eval_emb)
classes = clf_sen_LR.classes_

best_idx = proba.argmax(axis=1)
best_p = proba.max(axis=1)
pred = classes[best_idx]

THRESHOLD= 0.60
y_pred_thr = np.where(best_p < THRESHOLD, OTHER_LABEL, pred)

print("accuracy:", round(accuracy_score(y_true, y_pred_thr), 4))
print(classification_report(y_true, y_pred_thr, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred_thr, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

accuracy: 0.5
              precision    recall  f1-score   support

    Director     0.7153    0.7305    0.7228       141
      Junior     0.3618    0.6261    0.4586       230
        Lead     0.7098    0.4413    0.5442       460
  Management     0.8012    0.6459    0.7152       418
Professional     0.5903    0.3888    0.4688      1219
      Senior     0.1866    0.7353    0.2976       170

    accuracy                         0.5000      2638
   macro avg     0.5608    0.5947    0.5346      2638
weighted avg     0.6053    0.5000    0.5227      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,103,0,0,3,33,2
Junior,1,144,8,0,56,21
Lead,3,9,203,22,157,66
Management,34,15,15,270,59,25
Professional,1,213,60,40,474,431
Senior,2,17,0,2,24,125


##paraphrase-multilingual-mpnet-base-v2

###Trainingsplit 20/80

In [32]:
df = df_seniority
X_all = df_seniority["text"]
y_all = df_seniority["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_sen_1_LR = LogisticRegression(class_weight="balanced")
clf_sen_1_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_sen_1_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

Batches:   0%|          | 0/236 [00:00<?, ?it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

v2 eval accuracy: 0.9046
              precision    recall  f1-score   support

    Director     0.9596    0.9645    0.9620       197
      Junior     0.6210    0.9390    0.7476        82
        Lead     0.9214    0.9097    0.9155       709
  Management     0.7530    0.8278    0.7886       151
      Senior     0.9585    0.8956    0.9260       747

    accuracy                         0.9046      1886
   macro avg     0.8427    0.9073    0.8679      1886
weighted avg     0.9135    0.9046    0.9071      1886



In [33]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "seniority"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)


# --- predict ---
y_pred = clf_sen_1_LR.predict(X_final_eval_emb)
y_true = final_eval["seniority"].tolist()

# --- results ---
all_labels = sorted(set(clf_sen_1_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.3992
              precision    recall  f1-score   support

    Director     0.6528    0.8936    0.7545       141
      Junior     0.3351    0.8174    0.4753       230
        Lead     0.6110    0.6283    0.6195       460
  Management     0.6341    0.7464    0.6857       418
Professional     0.0000    0.0000    0.0000      1219
      Senior     0.1502    0.8118    0.2534       170

    accuracy                         0.3992      2638
   macro avg     0.3972    0.6496    0.4648      2638
weighted avg     0.2808    0.3992    0.3148      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,126,0,5,5,0,5
Junior,1,188,11,7,0,23
Lead,6,19,289,37,0,109
Management,44,12,14,312,0,36
Professional,11,328,146,126,0,608
Senior,5,14,8,5,0,138


###threshold to predict "Professional"

In [34]:
OTHER_LABEL = "Professional"

proba = clf_sen_1_LR.predict_proba(X_final_eval_emb)
classes = clf_sen_1_LR.classes_

best_idx = proba.argmax(axis=1)
best_p = proba.max(axis=1)
pred = classes[best_idx]

THRESHOLD= 0.60
y_pred_thr = np.where(best_p < THRESHOLD, OTHER_LABEL, pred)

print("accuracy:", round(accuracy_score(y_true, y_pred_thr), 4))
print(classification_report(y_true, y_pred_thr, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred_thr, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

accuracy: 0.5193
              precision    recall  f1-score   support

    Director     0.6964    0.8298    0.7573       141
      Junior     0.4339    0.7130    0.5395       230
        Lead     0.7743    0.4848    0.5963       460
  Management     0.8249    0.6986    0.7565       418
Professional     0.6184    0.3642    0.4584      1219
      Senior     0.1776    0.7647    0.2882       170

    accuracy                         0.5193      2638
   macro avg     0.5876    0.6425    0.5660      2638
weighted avg     0.6380    0.5193    0.5418      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,117,0,1,3,19,1
Junior,1,164,5,3,44,13
Lead,1,7,223,17,136,76
Management,41,0,2,292,54,29
Professional,5,195,56,36,444,483
Senior,3,12,1,3,21,130


###Trainingsplit 50/50

In [35]:
df = df_seniority
X_all = df_seniority["text"]
y_all = df_seniority["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.50,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_sen_2_LR = LogisticRegression(class_weight="balanced")
clf_sen_2_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_sen_2_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

Batches:   0%|          | 0/148 [00:00<?, ?it/s]

Batches:   0%|          | 0/148 [00:00<?, ?it/s]

v2 eval accuracy: 0.8891
              precision    recall  f1-score   support

    Director     0.9375    0.9146    0.9259       492
      Junior     0.5679    0.8976    0.6957       205
        Lead     0.9097    0.8979    0.9038      1773
  Management     0.7281    0.8360    0.7783       378
      Senior     0.9554    0.8837    0.9182      1866

    accuracy                         0.8891      4714
   macro avg     0.8197    0.8860    0.8444      4714
weighted avg     0.9013    0.8891    0.8927      4714



In [36]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "seniority"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)


# --- predict ---
y_pred = clf_sen_2_LR.predict(X_final_eval_emb)
y_true = final_eval["seniority"].tolist()

# --- results ---
all_labels = sorted(set(clf_sen_2_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.3995
              precision    recall  f1-score   support

    Director     0.6632    0.9078    0.7665       141
      Junior     0.3077    0.8174    0.4471       230
        Lead     0.6250    0.6304    0.6277       460
  Management     0.6479    0.7440    0.6927       418
Professional     0.0000    0.0000    0.0000      1219
      Senior     0.1539    0.8059    0.2585       170

    accuracy                         0.3995      2638
   macro avg     0.3996    0.6509    0.4654      2638
weighted avg     0.2838    0.3995    0.3158      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,128,0,4,4,0,5
Junior,1,188,13,6,0,22
Lead,8,30,290,33,0,99
Management,42,17,12,311,0,36
Professional,10,360,139,119,0,591
Senior,4,16,6,7,0,137


###threshold to predict "Professional"

In [37]:
OTHER_LABEL = "Professional"

proba = clf_sen_2_LR.predict_proba(X_final_eval_emb)
classes = clf_sen_2_LR.classes_

best_idx = proba.argmax(axis=1)
best_p = proba.max(axis=1)
pred = classes[best_idx]

THRESHOLD= 0.60
y_pred_thr = np.where(best_p < THRESHOLD, OTHER_LABEL, pred)

print("accuracy:", round(accuracy_score(y_true, y_pred_thr), 4))
print(classification_report(y_true, y_pred_thr, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred_thr, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

accuracy: 0.5394
              precision    recall  f1-score   support

    Director     0.7099    0.8156    0.7591       141
      Junior     0.4289    0.7087    0.5344       230
        Lead     0.7910    0.4609    0.5824       460
  Management     0.8584    0.6818    0.7600       418
Professional     0.6197    0.4290    0.5070      1219
      Senior     0.1917    0.7353    0.3041       170

    accuracy                         0.5394      2638
   macro avg     0.5999    0.6386    0.5745      2638
weighted avg     0.6480    0.5394    0.5630      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,115,0,1,3,21,1
Junior,1,163,6,2,50,8
Lead,1,6,212,15,157,69
Management,39,1,2,285,66,25
Professional,4,198,46,24,523,424
Senior,2,12,1,3,27,125


###Trainingsplit 60/40

In [38]:
df = df_seniority
X_all = df_seniority["text"]
y_all = df_seniority["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.60,
    random_state=42,
    stratify=y_all
)

# embeddings
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# training LR classifier on embeddings
clf_sen_3_LR = LogisticRegression(class_weight="balanced")
clf_sen_3_LR.fit(X_train_emb, y_train)

# eval on v2 split

y_pred = clf_sen_3_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Batches:   0%|          | 0/177 [00:00<?, ?it/s]

v2 eval accuracy: 0.8886
              precision    recall  f1-score   support

    Director     0.9413    0.9237    0.9324       590
      Junior     0.5827    0.9061    0.7093       245
        Lead     0.9081    0.9013    0.9047      2128
  Management     0.7232    0.8172    0.7673       454
      Senior     0.9513    0.8799    0.9142      2240

    accuracy                         0.8886      5657
   macro avg     0.8213    0.8857    0.8456      5657
weighted avg     0.8997    0.8886    0.8919      5657



In [39]:
# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "seniority"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)


# --- predict ---
y_pred = clf_sen_3_LR.predict(X_final_eval_emb)
y_true = final_eval["seniority"].tolist()

# --- results ---
all_labels = sorted(set(clf_sen_3_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.4011
              precision    recall  f1-score   support

    Director     0.6684    0.9149    0.7725       141
      Junior     0.2983    0.8261    0.4383       230
        Lead     0.6402    0.6304    0.6353       460
  Management     0.6389    0.7536    0.6915       418
Professional     0.0000    0.0000    0.0000      1219
      Senior     0.1555    0.7882    0.2597       170

    accuracy                         0.4011      2638
   macro avg     0.4002    0.6522    0.4662      2638
weighted avg     0.2846    0.4011    0.3166      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,129,0,4,5,0,3
Junior,1,190,13,6,0,20
Lead,9,33,290,32,0,96
Management,42,19,8,315,0,34
Professional,8,378,132,126,0,575
Senior,4,17,6,9,0,134


###threshold to predict "Professional"

In [40]:
OTHER_LABEL = "Professional"

proba = clf_sen_3_LR.predict_proba(X_final_eval_emb)
classes = clf_sen_3_LR.classes_

best_idx = proba.argmax(axis=1)
best_p = proba.max(axis=1)
pred = classes[best_idx]

THRESHOLD= 0.60
y_pred_thr = np.where(best_p < THRESHOLD, OTHER_LABEL, pred)

print("accuracy:", round(accuracy_score(y_true, y_pred_thr), 4))
print(classification_report(y_true, y_pred_thr, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred_thr, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

accuracy: 0.5368
              precision    recall  f1-score   support

    Director     0.7089    0.7943    0.7492       141
      Junior     0.4167    0.6957    0.5212       230
        Lead     0.8148    0.4304    0.5633       460
  Management     0.8777    0.6866    0.7705       418
Professional     0.6025    0.4389    0.5078      1219
      Senior     0.1944    0.7294    0.3069       170

    accuracy                         0.5368      2638
   macro avg     0.6025    0.6292    0.5698      2638
weighted avg     0.6463    0.5368    0.5602      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,112,0,1,2,25,1
Junior,1,160,4,2,57,6
Lead,1,6,198,8,180,67
Management,38,2,2,287,64,25
Professional,2,204,37,26,535,415
Senior,4,12,1,2,27,124
